<a href="https://colab.research.google.com/github/Ambaright/ST-554-Homework-7/blob/main/Baright_ST554_Homework_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Amanda Baright

Purpose: ST 554 Homework 7

Date: 03.31.2026

# Homework 7: Fitting MLR and Logistic Regression

This homework will serve as practice for fitting MLR and logistic regression models (including penalized and regularized models). We will use the data from the UCI Machine Learning Repository that investigates wine quality. More info on the data can be found [here](https://archive.ics.uci.edu/dataset/186/wine+quality).

We have a few different input variables to work with and the common output variable is `quality`. However, for the purpose of this homework, we will treat `alcohol` as our target variable when fitting an MLR model and when fitting a logistic regression model we will use the type of wine as the response variable.

## Loading in the Data

First we want to read in the data. The data is given in two csv files (`winequality-red.csv` and `winequality-white.csv`) which we willl want to combine into one dataset. We will then create a new variable that represents the type of wine (red or white).

Note: The two csv files were downloaded from the UCI Machine Learning Repository and then uploaded into the Google Colab environment.

In [60]:
# Packages for the assignment
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, RidgeCV, Ridge, ElasticNetCV, ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score

# Now read in the data
red_wine = pd.read_csv('winequality-red.csv', sep=';')
white_wine = pd.read_csv('winequality-white.csv', sep=';')

print("Red wine head: ")
print(red_wine.head())
print("Red wine shape: ", red_wine.shape)

print("\nWhite wine head: ")
print(white_wine.head())
print("White wine shape: ", white_wine.shape)

Red wine head: 
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2   

Now we wish to combine these two datasets into one wine dataset. Before we do that, we will add in a new variable called `type` into each dataset to identify which type of wine we are working with. Here we will denote red wine as `type = 0` and white wine as `type = 1`, so the red wine will become our reference level.

We then combine these two datasets into one large wine dataset.

In [30]:
red_wine['type'] = 0
white_wine['type'] = 1

# Combine the datasets
wine_df = pd.concat([red_wine, white_wine], axis=0).reset_index(drop=True)
wine_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


We might want to drop any rows with missing values as a form of data cleaning.

In [31]:
wine_df.dropna(inplace=True)

## Split the Data

We now want to split the data into a training and test set, using stratified sampling to make sure there is a similar proportion of white and red wines in the training and test sets. To complete the stratified sampling we can use the option `stratify = wine_df['type']` in our `train_test_split` function.

In [32]:
train_df, test_df = train_test_split(
    wine_df,
    test_size=0.2,
    stratify=wine_df['type'],
    random_state=42
)

# Double check the shape of the training and test set
print("Training set shape: ", train_df.shape)
print(train_df.head())

print("\nTest set shape: ", test_df.shape)
print(test_df.head())

Training set shape:  (5197, 13)
      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
4110            6.2              0.30         0.49            11.2      0.058   
3062            8.3              0.19         0.49             1.2      0.051   
5236            8.7              0.30         0.34             4.8      0.018   
497             7.2              0.34         0.32             2.5      0.090   
826             7.5              0.27         0.34             2.3      0.050   

      free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
4110                 68.0                 215.0  0.99656  3.19       0.60   
3062                 11.0                 137.0  0.99180  3.06       0.46   
5236                 23.0                 127.0  0.99474  3.12       0.49   
497                  43.0                 113.0  0.99660  3.32       0.79   
826                   4.0                   8.0  0.99510  3.40       0.64   

      alcohol  qua

## Regression Task (`alcohol` as Response)

Now we will work on a few regression tasks where we use `alcohol` as a response. The first thing we will want to do is differentiate between our predictors versus our response in the training and test sets.

In [33]:
X_train = train_df.drop("alcohol", axis = 1)
y_train = train_df["alcohol"]

X_test = test_df.drop("alcohol", axis = 1)
y_test = test_df["alcohol"]

We now want to scale the data since we are using regularized methods. This can be done by subtracting the mean and dividing by the sd. This is done for all of the `X_train` observations with a lambda function. We will then use the training means and sds for the test set too.

In [34]:
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0)

X_train = X_train.apply(lambda x: (x - np.mean(x))/np.std(x), axis = 0)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4110,-0.780839,-0.235499,1.194170,1.229111,0.054220,2.108751,1.749101,0.623768,-0.174702,0.465371,0.201651,0.571351
3062,0.833361,-0.903661,1.194170,-0.887164,-0.142398,-1.098649,0.374488,-0.953600,-0.986520,-0.480979,0.201651,0.571351
5236,1.140827,-0.235499,0.150449,-0.125305,-1.069314,-0.423407,0.198256,0.020657,-0.611835,-0.278190,1.341998,0.571351
497,-0.012173,0.007469,0.011287,-0.612048,0.953048,0.701997,-0.048470,0.637023,0.637117,1.749702,-0.938696,-1.750237
826,0.218427,-0.417725,0.150449,-0.654374,-0.170487,-1.492540,-1.898910,0.139954,1.136697,0.735756,1.341998,-1.750237


We then want to apply the training means and stds when standardizing the test set.

In [35]:
# quick function to standardize based on supplied means and stds
def my_std_fun(x, means, stds):
    return((x-means)/stds)
#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])
X_test.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
1739,-0.703973,-0.174757,0.150449,-0.675536,-0.310928,-0.592217,-0.682907,-0.655358,0.512221,-0.683768,-0.938696,0.571351
2845,0.602761,-0.842919,0.567938,-0.040654,-0.030045,0.589456,0.903185,0.206230,-0.237149,-0.886557,1.341998,0.571351
5282,-0.319639,-1.146628,0.637519,1.588878,-0.339017,2.755858,1.185157,0.908755,0.137536,1.682105,0.201651,0.571351
5822,-1.472639,0.554146,-1.449922,-0.908327,-0.760342,-1.380000,-1.141111,-1.298235,1.823620,-0.345786,-2.079043,0.571351
5237,-0.627106,-0.296241,0.011287,-0.633211,-1.181667,0.195565,-0.471428,-1.523573,0.137536,0.870949,1.341998,0.571351


Now that we did our standardization on the Training and Test sets we can finally start looking at some modeling.

### Multiple Linear Regression Models

We first want to fit four different MLR models. Even though we don’t need to use CV on these models to determine a tuning parameter, we can use CV on those to determine the best MLR model (using the training set alone). Here we use negative mean squared error as our metric.

#### Model 1: Full MLR model

Our first model will be a basic MLR where we use all of the predictors and the `alcohol` response. We can then print out all of the coefficient estimates.

In [39]:
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

#### Model 2: MLR with Five Predictors

The next model will look at five predictors (`fixed acidity`, `volatile acidity`, `citric acid`, `pH`, and `type`).

In [38]:
cv_mlr2= cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "volatile acidity", "citric acid", "pH", "type"]],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

#### Model 3: Interaction Term

Now we want a model that includes some interaction terms. Here we might want to look at the interaction of `density` and `pH`.

In [41]:
interaction = PolynomialFeatures(interaction_only=True, include_bias = False)
design_interaction = interaction.fit_transform(X_train[["density", "pH"]])
cv_mlr3 = cross_validate(
    LinearRegression(),
    design_interaction,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

#### Model 4: Polynomial Terms

Now we want to include some polynomial terms. Here we might want to look at a two degree polynomial for `residual sugar` and `density`.

In [42]:
poly = PolynomialFeatures(degree = 2, include_bias = False)
design_poly = poly.fit_transform(X_train[["residual sugar", "density"]])
cv_mlr4 = cross_validate(
    LinearRegression(),
    design_poly,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

Now we need to select our best MLR model using the model metrics. The best model will have the smallest value. Here we see that the full model actually has the best performance out of the four MLR models.

In [43]:
print(np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])),
      np.sqrt(-sum(cv_mlr4['test_score'])))


1.1548944121374503 2.656016913605667 1.9067439275237186 1.6175760096537062


Here we can examine the coefficient estimates of the best MLR model.

In [44]:
mlr_best = LinearRegression().fit(X_train, y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.489583092809905
[['fixed acidity' '0.6628226753046493']
 ['volatile acidity' '0.13659555386556077']
 ['citric acid' '0.07698578383284707']
 ['residual sugar' '1.0680497388874999']
 ['chlorides' '-0.038223319373377904']
 ['free sulfur dioxide' '-0.05550544789657755']
 ['total sulfur dioxide' '-0.01624484064385051']
 ['density' '-1.9543140082320571']
 ['pH' '0.41337475834032245']
 ['sulphates' '0.15157121014222186']
 ['quality' '0.08870369438131037']
 ['type' '-0.4732831467916546']]


### LASSO Model

Now we want to fit a LASSO model with a set of predictors of our choosing, here we try by using all of the predictors as LASSO has some model selection built in.

In [45]:
lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_train,
         y_train)

We then will use CV to select the tuning parameter. Before we do that, we might want to look at all of the different alpha values and the CV errors.

In [46]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.002 , 0.2665],
       [0.0019, 0.2665],
       [0.0022, 0.2665],
       [0.0018, 0.2665],
       [0.0017, 0.2665],
       [0.0015, 0.2665],
       [0.0024, 0.2665],
       [0.0014, 0.2665],
       [0.0013, 0.2666],
       [0.0013, 0.2666]])

Here we see that our best alpha value might be 0.002. We can double check this, as well as look at our coefficient estimates.

In [47]:
print(lasso_mod.alpha_)
print(lasso_mod.intercept_)
print(np.array(list(zip(X_train.columns, lasso_mod.coef_))))

0.002045005162739157
10.489583092809905
[['fixed acidity' '0.6491411016953442']
 ['volatile acidity' '0.13427489553962807']
 ['citric acid' '0.07425788238382583']
 ['residual sugar' '1.0405207584315583']
 ['chlorides' '-0.0376389627954318']
 ['free sulfur dioxide' '-0.05251889158702972']
 ['total sulfur dioxide' '-0.020574908363117726']
 ['density' '-1.922713131396445']
 ['pH' '0.4041871381349288']
 ['sulphates' '0.14820724643522276']
 ['quality' '0.09232334928255688']
 ['type' '-0.4600966177138631']]


Our cross validation does yeild a best tuning parameter of 0.002. We can then fit our best LASSO model with this tuning parameter.

In [50]:
lasso_best = Lasso(lasso_mod.alpha_).fit(X_train,y_train)

### Ridge Regression

We now want to find the best Ridge Regression model using CV, with a set of parameters of our choosing.

In [51]:
ridge_mod = RidgeCV(cv=5) \
    .fit(X_train,
         y_train)

We then might want to look at the best tuning parameter from the CV and the coefficient estimates.

In [55]:
print(ridge_mod.alpha_)
print(ridge_mod.intercept_)
print(np.array(list(zip(X_train.columns, ridge_mod.coef_))))

10.0
10.489583092809907
[['fixed acidity' '0.6443283643574967']
 ['volatile acidity' '0.1394328047213727']
 ['citric acid' '0.07894732615801496']
 ['residual sugar' '1.034946115962217']
 ['chlorides' '-0.043010420457991236']
 ['free sulfur dioxide' '-0.052996907688167776']
 ['total sulfur dioxide' '-0.025179135533601335']
 ['density' '-1.912673957213775']
 ['pH' '0.4033652153939439']
 ['sulphates' '0.1498755804162559']
 ['quality' '0.095624039100916']
 ['type' '-0.4544109184767501']]


We then want to fit our best Ridge model, that has an alpha of 10.

In [61]:
ridge_best = Ridge(ridge_mod.alpha_).fit(X_train,y_train)